<a href="https://colab.research.google.com/github/jackgomez27-sys/OTI-unsch/blob/main/C%C3%B3digo_clases.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from abc import ABC, abstractmethod
from datetime import date
from enum import Enum

# ==========================================
# 1. CLASES ABSTRACTAS Y HERENCIA
# ==========================================

class EntidadConNombre(ABC):
    def __init__(self, nombre: str, max_length: int):
        self._max_length = max_length
        self.nombre = nombre  # Llama al setter para validar

    @property
    def nombre(self):
        return self._nombre

    @nombre.setter
    def nombre(self, valor: str):
        if not valor or len(valor.strip()) == 0:
            raise ValueError("El nombre no puede quedar vacío.")
        if len(valor) > self._max_length:
            raise ValueError(f"El nombre no puede exceder los {self._max_length} caracteres.")
        self._nombre = valor

    @abstractmethod
    def resumen(self) -> str:
        pass


class Persona(ABC):
    def __init__(self, dni: str, nombres: str, apellidos: str):
        self.dni = dni
        self.nombres = nombres
        self.apellidos = apellidos

    def nombre_completo(self) -> str:
        return f"{self.nombres} {self.apellidos}"

    @abstractmethod
    def rol(self) -> str:
        pass


class Estudiante(Persona):
    def __init__(self, dni: str, nombres: str, apellidos: str, codigo_estudiante: str, codigo_escuela: str):
        super().__init__(dni, nombres, apellidos)
        self.codigo_estudiante = codigo_estudiante
        self.codigo_escuela = codigo_escuela

    def rol(self) -> str:
        return "Estudiante"


class Docente(Persona):
    def __init__(self, dni: str, nombres: str, apellidos: str, codigo_docente: str):
        super().__init__(dni, nombres, apellidos)
        self.codigo_docente = codigo_docente

    def rol(self) -> str:
        return "Docente"

# ==========================================
# 2. CLASES DE DOMINIO
# ==========================================

class Curso(EntidadConNombre):
    _generador_id = 1

    def __init__(self, codigo: str, nombre: str, creditos: float, ht: int, hp: int, hs: int, hv: int):
        super().__init__(nombre, 200)
        self._id = f"CURSO-{Curso._generador_id:04d}"
        Curso._generador_id += 1

        self.codigo = codigo
        self.creditos = creditos
        self.horas_teoricas = ht
        self.horas_practicas = hp
        self.horas_seminario = hs
        self.horas_virtuales = hv

    @property
    def id(self): return self._id  # Solo lectura, nadie puede cambiarlo

    @property
    def creditos(self): return self._creditos

    @creditos.setter
    def creditos(self, valor: float):
        if valor <= 0:
            raise ValueError("Los créditos deben ser mayores que cero.")
        self._creditos = round(valor, 2)

    def _validar_horas(self, valor: int, tipo: str):
        if valor < 0 or valor > 255:
            raise ValueError(f"Las horas {tipo} deben estar entre 0 y 255.")
        return valor

    @property
    def horas_teoricas(self): return self._ht
    @horas_teoricas.setter
    def horas_teoricas(self, v): self._ht = self._validar_horas(v, "teóricas")

    @property
    def horas_practicas(self): return self._hp
    @horas_practicas.setter
    def horas_practicas(self, v): self._hp = self._validar_horas(v, "prácticas")

    @property
    def horas_seminario(self): return self._hs
    @horas_seminario.setter
    def horas_seminario(self, v): self._hs = self._validar_horas(v, "de seminario")

    @property
    def horas_virtuales(self): return self._hv
    @horas_virtuales.setter
    def horas_virtuales(self, v): self._hv = self._validar_horas(v, "virtuales")

    @property
    def horas_totales(self) -> int:
        return self._ht + self._hp + self._hs + self._hv  # Calculado, no guardado

    def resumen(self) -> str:
        return f"[Curso] {self.codigo} - {self.nombre} ({self.horas_totales} hrs)"


class Seccion:
    def __init__(self, codigo: str, codigo_curso: str, codigo_periodo: str, vacantes: int):
        self.codigo = codigo
        self.codigo_curso = codigo_curso
        self.codigo_periodo = codigo_periodo
        self._matriculados = 0
        self.vacantes = vacantes

    @property
    def vacantes(self): return self._vacantes

    @vacantes.setter
    def vacantes(self, valor: int):
        if valor <= 0:
            raise ValueError("Las vacantes deben ser mayores que cero.")
        if hasattr(self, '_matriculados') and valor < self._matriculados:
            raise ValueError("Las vacantes no pueden ser menores a los matriculados actuales.")
        self._vacantes = valor

    @property
    def matriculados(self): return self._matriculados  # Solo lectura

    @property
    def vacantes_disponibles(self) -> int:
        return self.vacantes - self.matriculados

    def matricular_estudiante(self):
        if self.vacantes_disponibles <= 0:
            raise ValueError("Matrícula rechazada: la sección está llena.")
        self._matriculados += 1

    def retirar_estudiante(self):
        if self._matriculados > 0:
            self._matriculados -= 1


class PlanEstudio(EntidadConNombre):
    def __init__(self, nombre: str, anio: int, obligatorios: float, electivos: float, extracurriculares: float, vigente: int):
        super().__init__(nombre, 36)
        self.anio_aprobacion = anio
        self.creditos_obligatorios = self._validar_cred(obligatorios)
        self.creditos_electivos = self._validar_cred(electivos)
        self.creditos_extracurriculares = self._validar_cred(extracurriculares)
        self._vigente = vigente

    def _validar_cred(self, valor: float):
        if valor < 0:
            raise ValueError("Los créditos del plan no pueden ser negativos.")
        return valor

    @property
    def total_creditos(self) -> float:
        return self.creditos_obligatorios + self.creditos_electivos + self.creditos_extracurriculares

    @property
    def vigente(self) -> bool:
        return bool(self._vigente)

    def resumen(self) -> str:
        return f"[Plan] {self.nombre} ({self.anio_aprobacion}) - {self.total_creditos} crs totales"


class EscuelaProfesional(EntidadConNombre):
    def __init__(self, codigo: str, nombre: str, info: str, perfil: str, dir_cod: str, sec_cod: str):
        super().__init__(nombre, 100)
        self.codigo = codigo
        self.informacion_general = info
        self.perfil_egresado = perfil
        self.codigo_director = dir_cod
        self.codigo_secretario = sec_cod

    def resumen(self) -> str:
        return f"[Escuela] {self.nombre}"


class EstadoPeriodo(Enum):
    PLANIFICADO = 1
    EN_CURSO = 2
    MATRICULA_CERRADA = 3
    FINALIZADO = 4

class PeriodoAcademico(EntidadConNombre):
    def __init__(self, nombre: str, numero: int, anio: int, f_ini: date, f_fin: date, verano: int, nota_min: float, estado: EstadoPeriodo):
        super().__init__(nombre, 45)
        self.numero = numero
        self.anio = anio
        self.fecha_inicio = f_ini
        self.fecha_fin = f_fin
        self._es_verano = verano
        self.nota_minima = nota_min
        self.estado = estado
        self._inicio_matricula = None
        self._fin_matricula = None

    @property
    def nota_minima(self): return self._nota_minima

    @nota_minima.setter
    def nota_minima(self, valor: float):
        if valor < 0 or valor > 20:
            raise ValueError("La nota mínima debe estar entre 0 y 20.")
        self._nota_minima = valor

    def establecer_fechas_matricula(self, inicio: date, fin: date):
        if fin < inicio:
            raise ValueError("La fecha de fin no puede ser anterior a la de inicio.")
        self._inicio_matricula = inicio
        self._fin_matricula = fin

    @property
    def es_verano(self) -> bool:
        return bool(self._es_verano)

    def resumen(self) -> str:
        return f"[Periodo] {self.nombre} - Estado: {self.estado.name}"


# ==========================================
# 3. PROGRAMA DE PRUEBA (Validación de Reglas)
# ==========================================

if __name__ == "__main__":
    print("--- PRUEBAS DE REGLAS DE LA OTI ---\n")

    # Regla 1: Validar horas de curso
    try:
        print("Intentando crear curso con horas negativas...")
        curso_malo = Curso("IS-101", "Física", 3.0, -5, 20, 0, 0)
    except ValueError as e:
        print(f"✔️ RECHAZADO: {e}")

    curso_ok = Curso("IS-282", "Ingeniería de Software", 4.0, 40, 20, 0, 0)
    print(f"✔️ ACEPTADO: Curso creado. Horas totales calculadas: {curso_ok.horas_totales}\n")

    # Regla 2: Validar vacantes y matriculados
    seccion = Seccion("SEC-1", "IS-282", "2026-II", 1)
    seccion.matricular_estudiante()
    try:
        print("Intentando matricular en sección llena...")
        seccion.matricular_estudiante()
    except ValueError as e:
        print(f"✔️ RECHAZADO: {e}")

    try:
        print("Intentando reducir vacantes por debajo de los matriculados...")
        seccion.vacantes = 0
    except ValueError as e:
        print(f"✔️ RECHAZADO: {e}\n")

    # Regla 3: Validar Fechas del periodo
    periodo_ok = PeriodoAcademico("Periodo 2026-II", 2, 2026, date(2026,8,1), date(2026,12,15), 0, 10.5, EstadoPeriodo.PLANIFICADO)
    try:
        print("Intentando asignar fin de matrícula antes del inicio...")
        periodo_ok.establecer_fechas_matricula(date(2026,7,20), date(2026,7,10))
    except ValueError as e:
        print(f"✔️ RECHAZADO: {e}\n")

    # Regla 4: Estado del Periodo (Evitar el número 7)
    try:
        print("Intentando crear periodo con estado '7'...")
        # Al exigir el tipo EstadoPeriodo, Python arrojará error si se pasa un 7 directo
        periodo_malo = PeriodoAcademico("Raro", 1, 2026, date.today(), date.today(), 0, 11, 7)
    except Exception:
        print("✔️ RECHAZADO: El sistema exige usar el Enum EstadoPeriodo (PLANIFICADO, EN_CURSO, etc).\n")

    # Regla 5: Polimorfismo
    print("--- RECORRIDO POLIMÓRFICO DE RESÚMENES ---")
    lista = [
        curso_ok,
        PlanEstudio("Plan Sistemas", 2018, 160, 30, 10, 1),
        EscuelaProfesional("EPIS", "Ing. Sistemas", "Info", "Perfil", "D1", "S1"),
        periodo_ok
    ]
    for item in lista:
        print(item.resumen())